In [ ]:
# ==========================================
# BLOCK 1: ENVIRONMENT SETUP & PERSISTENCE
# ==========================================
# @title ⚙️ 1. Setup Environment & Google Drive

DRIVE_WORKSPACE_NAME = "AutoScribe_Workspace" # @param {type:"string"}
SHARED_DRIVE_NAME = "" # @param {type:"string"}

import os
import sys
from datetime import datetime
from google.colab import drive
from IPython.display import clear_output, display
from IPython.utils import capture
import ipywidgets as widgets

def log_event(level, message, print_to_console=True):
    if print_to_console: print(message)
    try:
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        with open(LOG_FILE, 'a', encoding='utf-8') as f:
            f.write(f"[{timestamp}] [{level}] {message}\n")
    except Exception: pass

def inf(msg, style, wdth):
    display(widgets.Button(description=msg, disabled=True, button_style=style, layout=widgets.Layout(min_width=wdth)))

print("Mounting Google Drive...")
drive.mount('/content/drive', force_remount=True)

if SHARED_DRIVE_NAME != "" and os.path.exists(f"/content/drive/Shareddrives/{SHARED_DRIVE_NAME}"):
    root_path = f"/content/drive/Shareddrives/{SHARED_DRIVE_NAME}"
else:
    root_path = "/content/drive/MyDrive"

workspace_name = DRIVE_WORKSPACE_NAME.strip() or "AutoScribe_Workspace"
BASE_DIR = f"{root_path}/{workspace_name}"
CACHE_DIR = os.path.join(BASE_DIR, "cache")
PIP_CACHE = os.path.join(CACHE_DIR, "pip_packages")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")
TEMP_DIR = "/content/temp_media"
MARKER_FILE = os.path.join(CACHE_DIR, ".setup_complete")
LOG_FILE = os.path.join(OUTPUT_DIR, "autoscribe_debug.log")

for directory in [PIP_CACHE, OUTPUT_DIR, TEMP_DIR]:
    os.makedirs(directory, exist_ok=True)

log_event("INFO", "=== NEW AUTOSCRIBE SESSION STARTED ===", print_to_console=False)

os.environ["HF_HOME"] = "/content/local_hf_cache"
sys.path.insert(0, PIP_CACHE)

with capture.capture_output() as cap:
    if not os.path.exists(MARKER_FILE):
        log_event("INFO", "First time setup: Installing all dependencies to Drive...", print_to_console=False)
        !pip install -q --target=$PIP_CACHE yt-dlp faster-whisper ffmpeg-python transformers accelerate torch torchvision
        with open(MARKER_FILE, 'w') as f: f.write("Setup complete.")
    else:
        log_event("INFO", "Base dependencies loaded from cache.", print_to_console=False)
    
    # ALWAYS upgrade yt-dlp to bypass latest YouTube bot-protections
    # --no-deps prevents massive I/O freezes on Drive
    log_event("INFO", "Checking for latest yt-dlp updates...", print_to_console=False)
    !pip install -q --target=$PIP_CACHE --upgrade --no-deps yt-dlp

BLOCK_1_COMPLETED = True
clear_output()
inf('\u2714 Workspace Ready & Logging Active', 'success', '350px')

In [ ]:
# ==========================================
# BLOCK 2: INGESTION & ROUTING
# ==========================================
# @title 📥 2. Source Configuration & Routing

SOURCE_TYPE = "URL" # @param ["URL", "Google_Drive", "Local_Upload"]
MEDIA_INPUT = "https://youtube.com/playlist?list=..." # @param {type:"string"}
WHISPER_MODE = "Auto" # @param ["On", "Off", "Auto"]
VISION_FALLBACK = True # @param {type:"boolean"}

if 'BLOCK_1_COMPLETED' not in locals():
    raise RuntimeError("🚨 ERROR: Workspace not initialized. Please run Block 1 first.")

import yt_dlp
import subprocess
import shutil
import os
from google.colab import files

routing_queue = []

def analyze_and_route(file_path, title, video_id):
    log_event("INFO", f"Analyzing audio stream for: {title}...")
    command = ["ffprobe", "-v", "error", "-select_streams", "a", "-show_entries", "stream=codec_type", "-of", "default=nw=1:nk=1", file_path]
    requires_vision = False
    
    try:
        result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        has_audio = "audio" in result.stdout.strip().lower()
        
        if WHISPER_MODE == "Auto":
            if not has_audio and VISION_FALLBACK:
                requires_vision = True
                log_event("INFO", "[ROUTING]: No audio stream detected. Tagged for Vision.")
            else:
                log_event("INFO", "[ROUTING]: Audio stream intact. Tagged for Whisper.")
        elif WHISPER_MODE == "Off" and VISION_FALLBACK:
            requires_vision = True
            log_event("INFO", "[ROUTING]: Whisper forced OFF. Tagged for Vision.")
            
    except Exception as e:
        log_event("ERROR", f"Audio analysis failed for {title}: {e}. Fallback to Vision.")
        requires_vision = True

    routing_queue.append({'id': video_id, 'title': title, 'path': file_path, 'use_vision': requires_vision})

log_event("INFO", f"Processing source type: {SOURCE_TYPE}")

if SOURCE_TYPE == "Local_Upload":
    uploaded = files.upload()
    for filename in uploaded.keys():
        dest_path = os.path.join(TEMP_DIR, filename)
        shutil.move(filename, dest_path)
        analyze_and_route(dest_path, filename, filename.split('.')[0])

elif SOURCE_TYPE == "Google_Drive":
    if os.path.exists(MEDIA_INPUT):
        filename = os.path.basename(MEDIA_INPUT)
        dest_path = os.path.join(TEMP_DIR, filename)
        shutil.copy2(MEDIA_INPUT, dest_path)
        analyze_and_route(dest_path, filename, filename.split('.')[0])
    else:
        log_event("ERROR", "File not found on Google Drive.")

elif SOURCE_TYPE == "URL":
    ydl_opts_meta = {'extract_flat': 'in_playlist', 'quiet': True, 'js_runtimes': {'nodejs': {}, 'node': {}}}
    with yt_dlp.YoutubeDL(ydl_opts_meta) as ydl:
        try:
            info = ydl.extract_info(MEDIA_INPUT, download=False)
            entries = info['entries'] if 'entries' in info else [info]
            for entry in entries:
                v_id = entry['id']
                title = entry.get('title', v_id)
                url = entry.get('url', MEDIA_INPUT)
                log_event("INFO", f"--- Downloading: {title} ---")
                
                ydl_opts_dl = {
                    'format': 'bestaudio/best',
                    'outtmpl': f'{TEMP_DIR}/{v_id}.%(ext)s',
                    'trim_file_name': 200,
                    'quiet': True,
                    'js_runtimes': {'nodejs': {}, 'node': {}}
                }
                with yt_dlp.YoutubeDL(ydl_opts_dl) as ydl_dl:
                    try:
                        dl_info = ydl_dl.extract_info(url, download=True)
                        local_file = ydl_dl.prepare_filename(dl_info)
                        analyze_and_route(local_file, title, v_id)
                    except Exception as e:
                        log_event("ERROR", f"Failed to download {title}: {e}")
        except Exception as e:
            log_event("ERROR", f"Failed to parse URL/Playlist: {e}")

BLOCK_2_COMPLETED = True
log_event("INFO", f"✅ Block 2 Complete. {len(routing_queue)} item(s) in queue.")

In [ ]:
# ==========================================
# BLOCK 3: AI PROCESSING & STREAM-TO-DRIVE
# ==========================================
# @title 🧠 3. Execute AI Transcription & Vision Analysis

import gc
import torch
import subprocess
import os
import sys
import re
from datetime import datetime
from PIL import Image

if 'BLOCK_2_COMPLETED' not in locals():
    raise RuntimeError("🚨 ERROR: System memory lost. Please run Block 1 and 2.")
if not routing_queue:
    raise ValueError("🚨 ERROR: Routing queue is empty.")

try:
    from faster_whisper import WhisperModel
except ImportError:
    raise RuntimeError("🚨 ERROR: AI Libraries not found. Run Block 1 to restore paths.")

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
    log_event("INFO", "✅ Hugging Face token authenticated.")
except Exception:
    error_msg = "🚨 CRITICAL ERROR: 'HF_TOKEN' not found in Colab Secrets."
    log_event("ERROR", error_msg)
    raise PermissionError(error_msg)

LOCAL_MODEL_CACHE = "/content/local_hf_cache"
os.makedirs(LOCAL_MODEL_CACHE, exist_ok=True)
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

# Create a unique export directory for this run
run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
EXPORT_DIR = os.path.join(OUTPUT_DIR, f"AutoScribe_Run_{run_timestamp}")
os.makedirs(EXPORT_DIR, exist_ok=True)

audio_tasks = [t for t in routing_queue if not t['use_vision']]
vision_tasks = [t for t in routing_queue if t['use_vision']]

def get_safe_filename(title):
    safe_title = re.sub(r'[^\w\s-]', '', title).strip().replace(' ', '_')
    return safe_title[:50]

if audio_tasks:
    log_event("INFO", "\n--- Loading Whisper Model (large-v3) ---")
    try:
        whisper_model = WhisperModel("large-v3", device="cuda", compute_type="float16", download_root=LOCAL_MODEL_CACHE)
        for task in audio_tasks:
            log_event("INFO", f"🎙️ Transcribing: {task['title']} (Streaming directly to Drive)")
            safe_name = get_safe_filename(task['title'])
            export_path = os.path.join(EXPORT_DIR, f"{safe_name}_audio.md")
            
            try:
                segments, info = whisper_model.transcribe(task['path'], beam_size=5, language="sv", condition_on_previous_text=False)
                with open(export_path, "w", encoding="utf-8") as f:
                    f.write(f"# Source: {task['title']}\n**Analysis Type:** Audio Transcription\n\n")
                    for s in segments:
                        line = f"[{divmod(int(s.start), 60)[0]:02d}:{divmod(int(s.start), 60)[1]:02d}] {s.text.strip()}"
                        f.write(line + "\n")
                        f.flush() # Stream to Drive instantly
            except Exception as e: log_event("ERROR", f"Whisper failed: {e}")
    except Exception as e: log_event("ERROR", f"Model load failed: {e}")
    finally:
        if 'whisper_model' in locals(): del whisper_model
        gc.collect()
        torch.cuda.empty_cache()

if vision_tasks:
    log_event("INFO", "\n--- Loading Moondream2 Vision Model ---")
    try:
        from transformers import AutoModelForCausalLM, AutoTokenizer
        model_id = "vikhyatk/moondream2"
        tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True, cache_dir=LOCAL_MODEL_CACHE)
        moondream = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True, cache_dir=LOCAL_MODEL_CACHE).to("cuda")
        moondream.eval()

        for task in vision_tasks:
            log_event("INFO", f"👁️ Processing Vision: {task['title']} (Streaming directly to Drive)")
            safe_name = get_safe_filename(task['title'])
            export_path = os.path.join(EXPORT_DIR, f"{safe_name}_vision.md")
            
            frame_dir = os.path.join(TEMP_DIR, f"frames_{task['id']}")
            os.makedirs(frame_dir, exist_ok=True)
            subprocess.run(["ffmpeg", "-i", task['path'], "-vf", "fps=1/30", f"{frame_dir}/frame_%04d.jpg", "-hide_banner", "-loglevel", "error"])
            
            frames = sorted(os.listdir(frame_dir))
            
            with open(export_path, "w", encoding="utf-8") as f:
                f.write(f"# Source: {task['title']}\n**Analysis Type:** Visual Analysis\n\n")
                for i, frame_file in enumerate(frames):
                    try:
                        img_path = os.path.join(frame_dir, frame_file)
                        enc_image = moondream.encode_image(Image.open(img_path))
                        answer = moondream.answer_question(enc_image, "Describe the key action or text in this frame concisely.", tokenizer)
                        line = f"[{divmod(i * 30, 60)[0]:02d}:{divmod(i * 30, 60)[1]:02d}] {answer}"
                        f.write(line + "\n")
                        f.flush() # Stream to Drive instantly
                    except Exception as e: log_event("WARNING", f"Skipping frame: {e}")
                    
    except Exception as e: log_event("ERROR", f"Vision Model failed: {e}")
    finally:
        if 'moondream' in locals(): del moondream
        gc.collect()
        torch.cuda.empty_cache()

BLOCK_3_COMPLETED = True
log_event("INFO", f"✅ Block 3 Complete. Output saved to: {EXPORT_DIR}")

In [ ]:
# ==========================================
# BLOCK 4: CLEANUP & SHUTDOWN
# ==========================================
# @title 📝 4. Finalize & Disconnect Runtime

import shutil
from google.colab import runtime

if 'BLOCK_3_COMPLETED' not in locals():
    raise RuntimeError("🚨 ERROR: Cannot finalize. Block 3 must run successfully first.")

log_event("INFO", "--- Cleaning up temporary files ---")
try:
    if os.path.exists(TEMP_DIR):
        shutil.rmtree(TEMP_DIR)
        log_event("INFO", "🗑️ Ephemeral media storage cleared.")
except Exception as e:
    log_event("WARNING", f"Cleanup warning: {e}")

log_event("INFO", "🎉 All processing complete. Your files are safe on Google Drive.")
log_event("INFO", "Disconnecting runtime to save Compute Units.")

# Disconnect the runtime
runtime.unassign()